# API потребителя /mcp/kb

Пути, команды, замена строк в секции (`replace_lines`), профили lite/std/full, настройки подключения (lang, off, limit) и коды ошибок endpoint-а /mcp/kb. Читать при подключении клиента или правке набора команд.

## Пути

| Путь | Что это | `get` возвращает |
|---|---|---|
| `""` (пустой путь) | корень базы | список областей (путь + название) |
| `coding/python` | тема | шапку темы + её содержимое: подтемы и статьи (путь + название) |
| `coding/python/asyncio` | статья | **оглавление**: шапку целиком + для остальных секций только путь и заголовок; с `full=true` — все секции целиком |
| `coding/python/asyncio#event-loop` | секция | одну секцию |
| `coding/python/asyncio#head` | шапка статьи | одну секцию |

- Отдельного поля `id` в API нет: на вход и на выход — всегда полный путь.
- Сервер нормализует путь: убирает `.ipynb`, лишние `/`, приводит регистр — и сообщает `Note: path normalized to …`.
- «Не найдено» по секции — в ответе список секций статьи (путь + заголовок).
- В описаниях инструментов: «path копируй из ответов, не составляй сам» (сам составляется только путь нового объекта: путь родителя + новое имя).

## Команды

| Команда | Что делает |
|---|---|
| `index(path?)` | плоский список всех статей «путь → название» во всей базе или внутри темы `path`, на всю глубину; без описаний (std/full) |
| `get(path, with_tags?, full?)` | тема / статья / секция (см. «Пути»). Для статьи по умолчанию оглавление, `full=true` — всё целиком |
| `section_tags(path)` | уникальные тексты тегов секций внутри `path`. `path` **обязателен** — область, тема, подтема или статья; по всей базе теги не отдаются (`not_allowed`, в `options` — список областей) |
| `sections_by_tag(tag, path?, with_tags?)` | все секции с точным тегом |
| `search_section_tags(substrings, path?)` | поиск тегов по точному совпадению подстрок (без учёта регистра, **любая** из подстрок); `path` необязателен; ответ — уникальные тексты тегов |
| `search_section_tags_regex(pattern, path?)` | то же по регулярному выражению |
| `search(substrings, path?)` | поиск по точному совпадению подстрок в заголовках и текстах секций (без учёта регистра); находится секция, где есть **любая** из подстрок (ИЛИ; словоформы — через общую часть слова: `задач`); ответ — «путь секции — заголовок» |
| `search_regex(pattern, path?)` | то же по регулярному выражению; ошибка в выражении — `invalid_pattern` с текстом ошибки |
| `save_topic(path, title, content)` | **upsert** темы (область/тема/подтема): нет — создаёт с шапкой; есть — заменяет шапку |
| `save_article(path, title, content)` | **upsert** статьи: нет — создаёт с шапкой; есть — заменяет шапку. Родитель должен существовать |
| `save_section(path, title, content)` | **upsert** секции по пути `статья#id`: нет — создаёт **в конце статьи**; есть — заменяет заголовок и текст. `title` обязателен всегда |
| `replace_lines(path, old, new)` | заменить в тексте существующей секции блок строк `old` на `new`: правка, удаление (`new = ""`), дополнение. `old` должен встретиться ровно один раз (см. «Замена строк») |
| `add_section_tags(path, tags)` / `drop_section_tags(path, tags)` | теги секции, источник `mcp` |
| `delete(path)` | удалить секцию, статью или **пустую** тему. Шапку отдельно удалить нельзя. Откат — через git |

- Каждая запись ставит секции `indexed = false`, `hash`, `updated`, `changed_by = mcp`.
- Сервер принимает и привычные синонимы параметров: `text`/`query`/`q` ↔ `substrings`, `id` → `path`, `tag` → `tags`.

## Замена строк

`replace_lines(path, old, new)` — правка куска секции без пересылки всего текста (профили std и full; в lite нет — там только `save_section`).

- `path` — `статья#id` **существующей** секции, в том числе `#head`. Новую секцию так не создать; заголовок не меняется — только через `save_section`.
- `old` — одна или несколько строк подряд, **целиком**, как в тексте секции. Пробелы в конце строк при сравнении не важны; один завершающий перевод строки игнорируется. Можно передать строку (строки через `\n`) или список строк. Пустой `old` — `invalid_input`.
- `new` — строки на замену (обязателен). `""` — удалить блок `old`. Дописать — повторить в `new` опорную строку из `old` и добавить новые после неё.
- Результат: `OK updated <путь секции>`; если текст не изменился — `unchanged`.

### Ошибки
| `code` | Когда | `options` | `hint` |
|---|---|---|---|
| `ambiguous` | `old` встречается в секции несколько раз; ничего не записано | каждое совпадение: номер строки, строка перед ним и после | добавить соседние строки в `old` и `new`, или переписать секцию: `save_section("<путь>", "<заголовок>", ...)` |
| `not_found` | `old` — только часть строки | строки, где эта часть есть | передать строку целиком |
| `not_found` | строк `old` нет в секции | до 3 похожих строк | `get("<путь секции>")` и скопировать строки точно |
| `invalid_input` | путь без `#id`, нет `old` | — | правильная форма вызова |

Совпадения ищутся и перекрывающиеся: блок `a\na` в тексте `a\na\na` — два совпадения, `ambiguous`.

## Профили lite / std / full

Одна база, один endpoint `/mcp/kb` — три набора инструментов под разную силу модели. Профиль выбирается параметром подключения `profile`; адрес задаёт человек в конфиге клиента, модель его не меняет.

| Профиль | Для кого |
|---|---|
| **lite** — примитив | ультраслабые модели: минимум команд и параметров, один очевидный путь работы |
| **std** — средний | модели, которые справляются с флагами и тегами |
| **full** — полный (по умолчанию) | все команды |

Параметров, которых нет в профиле, модель не видит вовсе (их нет в описании инструмента); сервер подставляет значения по умолчанию.

| Команда | lite | std | full |
|---|---|---|---|
| `index` | — (навигация через `get`) | `index(path?)` | то же |
| `get` | `get(path)` → `with_tags=false`, `full=false` | `get(path, with_tags?, full?)` | то же |
| `search` | `search(text)` — **одна** подстрока | `search(substrings)` — несколько, любая | то же |
| `search_regex` | — | — | ✓ |
| `section_tags` | — | ✓ | ✓ |
| `search_section_tags` | — | ✓ | ✓ |
| `search_section_tags_regex` | — | — | ✓ |
| `sections_by_tag` | — | ✓ | ✓ |
| `save_topic` | — | ✓ | ✓ |
| `save_article` | ✓ (только в существующей теме) | ✓ | ✓ |
| `save_section` | ✓ | ✓ | ✓ |
| `replace_lines` | — (секция целиком через `save_section`) | ✓ | ✓ |
| `add_section_tags` | — | ✓ | ✓ |
| `drop_section_tags` | — | — | ✓ |
| `delete` | — | — | ✓ |

**Умолчания по профилям:** `lang` — `en` везде; `limit` — lite 5, std 10, full 20.

**Путь работы в lite** (пишется в описания инструментов):
1. Найти: `search(text)` → пути секций; или по дереву: `get("")` → области, `get(тема)` → подтемы и статьи.
2. Прочитать: `get(путь статьи)` → оглавление; `get(путь секции)` → текст.
3. Записать или поправить: `save_section("статья#id", title, content)` — новая встанет в конец, существующая заменится.
4. Нет подходящей статьи — `save_article` в существующей теме.

Теги, структура тем и удаление в lite недоступны — это работа индексатора и более сильных клиентов.

## Настройки подключения

Задаются клиентом при подключении — параметрами адреса endpoint:
```
http://127.0.0.1:8888/mcp/kb?profile=lite&lang=ru&token=…
```

| Параметр | Значения | По умолчанию | Что делает |
|---|---|---|---|
| `profile` | `lite` / `std` / `full` | `full` | набор команд и параметров |
| `lang` | `en` / `ru` | `en` | язык служебных текстов и описаний инструментов |
| `limit` | число | по профилю | максимум вариантов в `options` ошибок |
| `off` | список через запятую | пусто (или `KB_MCP_OFF`) | отключить возможности MCP, с которыми клиент работает плохо |

- Разные клиенты (слабая локальная модель, Claude Code с ПК) работают с одной базой с разными настройками.
- Токен — общий с вебом Jupyter (см. статью «Запуск и подключение»).

### Язык (`lang`)
- По умолчанию служебные тексты (`message`, `hint`, описания инструментов и параметров) — **английские**.
- `lang=ru` переключает **всё** служебное на русский целиком, без смеси языков — для моделей, которые от обилия английского сами переходят на английский.
- Коды ошибок, имена команд и полей, пути — всегда латиница. Содержимое базы (заголовки, тексты, теги) не переводится.

### Отключаемые возможности (`off`)
| Значение | Что отключает |
|---|---|
| `structured` | `structuredContent` в ответах — модель видит только плоский текст `content` |
| `instructions` | поле `instructions` в ответе `initialize` |

- Зачем: Claude Code показывает модели `structuredContent` (JSON) вместо `content`, даже без `outputSchema`. Для него — `off=structured`.
- Неизвестные значения игнорируются. Умолчание для всех клиентов — переменная окружения сервера `KB_MCP_OFF`; параметр подключения его заменяет, `off=` (пусто) включает всё обратно.

## Ошибки и правила для слабых моделей

Цель: модель никогда не застревает. Любой ответ либо даёт готовые пути для следующего шага, либо объясняет, что сделать дальше, и предлагает варианты.

### Принципы
1. **Одна форма ответа.** Модель читает `content` — там всегда плоский текст (см. статью «Формат ответов MCP»). JSON для программ: успех `{"ok": true, "result": …}`, ошибка `{"ok": false, "error": {…}}`.
2. **Пути копируются, а не собираются.** Каждый ответ отдаёт канонические полные пути всего, о чём говорит. Ответ на запись — путь объекта, без повторения текста.
3. **Терпимость вместо ошибки.** Если намерение однозначно — сервер исправляет сам (лишний `.ipynb`, `/`, регистр, пробелы в имени) и сообщает `"normalized": "<канонический путь>"`, чтобы модель училась.
4. **Ошибка = подсказка + варианты:** `message` (что не так), `hint` (готовый вызов команды), `options` (что точно есть рядом — соседи по дереву, плюс простое сходство написания; «близость по смыслу» сервер не считает). Где вычислить нечего — `options` пустой, а `hint` даёт вызов поиска из профиля.
5. **Upsert говорит, что сделал:** `"action": "created"` / `"updated"` (`OK created …` / `OK updated …`). `created` там, где модель хотела правку, — сигнал опечатки в пути.
6. **Безвредное — не ошибка.** Добавить существующий тег, убрать отсутствующий — `ok` с `"unchanged": true`.
7. **Списки отдаются целиком** — без обрезки и постраничности; `limit` ограничивает только `options` в ошибках. Размер текста регулирует `full` у `get`.
8. **Без внутренностей:** нет слов «ячейка», «ноутбук», `.ipynb`, `metadata`, стектрейсов.
9. **Короткие фиксированные коды ошибок** — модель ветвится по `code`, не разбирая текст.

### Форма ошибки (JSON)
```json
{"ok": false, "error": {
  "code": "not_found",
  "message": "Section 'coding/python/asyncio#evnt-loop' not found.",
  "hint": "Pick a path from options, or call get(\"coding/python/asyncio\").",
  "options": [
    {"path": "coding/python/asyncio#event-loop", "title": "Event loop"},
    {"path": "coding/python/asyncio#tasks",      "title": "Задачи"}
  ]
}}
```
Примеры успеха:
```json
{"ok": true, "result": {"path": "coding/python/asyncio#event-loop", "action": "created"}}
{"ok": true, "result": {"path": "coding/python/asyncio"}, "normalized": "coding/python/asyncio"}
{"ok": true, "result": {"path": "coding/python/asyncio#event-loop", "unchanged": true}}
```

### Коды, options и hint
Сортировка `options`: сначала по сходству написания с запрошенным, затем по алфавиту; не больше `limit`.

| `code` | Когда | `options` | `hint` |
|---|---|---|---|
| `not_found` | нет секции | секции этой статьи | `get("<статья>")` |
| `not_found` | нет статьи | статьи ближайшей существующей темы | `get("<тема>")`; не та тема — `search(...)` |
| `not_found` | нет темы | дочерние темы ближайшего предка (или все области) | `get("<предок>")` или `get("")` |
| `not_found` | нет тега (`sections_by_tag`) | теги, содержащие запрошенный текст | `search_section_tags(["<часть тега>"])` |
| — | `search*` ничего не нашли — **не ошибка**: `ok` с пустым списком | — | укоротить подстроку, добавить варианты (`["задач", "task"]`), сузить/убрать `path` |
| `invalid_name` | имя или `id` не из `a-z0-9-` | исправленный вариант: нижний регистр, пробелы → `-`, транслит (`"Отмена задач"` → `otmena-zadach`) | повторить с вариантом |
| `duplicate_title` | заголовок уже есть у **другой** секции статьи (ловит опечатки в `id`) | секция с этим заголовком | `save_section("<её путь>", …)` или другой заголовок |
| `too_deep` | глубже 3-го уровня тем | тема 3-го уровня на этом пути | `save_article("<подтема>/<имя>", …)` |
| `not_allowed` | статья прямо в области | темы этой области | `save_article("<тема>/<имя>", …)` |
| `not_found` | запись в несуществующую тему / статью (родитель не создаётся сам) | как для «нет темы» / «нет статьи» | `get("<предок>")`; создать — `save_topic` / `save_article` |
| `not_allowed` | `section_tags` без `path` | все области | `section_tags("<область>")` |
| `not_allowed` | удалить шапку | путь статьи/темы | `delete("<статья>")` |
| `not_allowed` | правка секции без заголовка | — | `get("<статья>")` — выбрать видимую |
| `not_allowed` | команды нет в профиле | — | имя команды и профиль, где она есть |
| `not_empty` | `delete` непустой темы | дочерние темы и статьи | удалить содержимое или выбрать другую тему |
| `invalid_pattern` | неверный regex | — | текст ошибки разбора; `search` вместо regex |
| `invalid_input` | нет параметра, неверный тип, путь секции без `#id` | — | правильная форма вызова |
| `ambiguous` | `replace_lines`: строки `old` встречаются в секции несколько раз | каждое совпадение с соседними строками | добавить соседние строки или `save_section(...)` (см. «Замена строк») |
| `conflict` | (админка) не совпал `version` | — | `get_full("<статья>")` заново |
| `internal` | непредвиденный сбой сервера | — | — |

В lite `hint` ссылается только на команды lite: вместо `search_section_tags`, `section_tags`, `delete` — `search(text)` и `get(path)`.